# SQL Governance

This notebook demonstrates SQL-based governance techniques for protecting
sensitive healthcare information.

Since Unity Catalog Dynamic Data Masking requires Enterprise permissions,
SQL Views are used to simulate column-level masking.

In [0]:
%sql
SHOW TABLES IN db_healthcare_kl.gold;

In [0]:
%sql
SELECT *
FROM db_healthcare_kl.gold.patient_cohorts
LIMIT 5;

In [0]:
%sql

CREATE OR REPLACE VIEW db_healthcare_kl.gold.patient_cohort_masked AS

SELECT

CASE
    WHEN person_id IS NOT NULL
    THEN CONCAT('****', RIGHT(person_id,4))
    ELSE NULL
END AS person_id,

cohort_name

FROM db_healthcare_kl.gold.patient_cohorts;

In [0]:
%sql

SELECT *
FROM db_healthcare_kl.gold.patient_cohort_masked
LIMIT 10;

In [0]:
%sql

CREATE OR REPLACE VIEW db_healthcare_kl.omop.person_data_masked AS

SELECT
    person_id,

    gender_concept_id,

    CONCAT(
        SUBSTRING(person_name,1,1),
        REPEAT('*', LENGTH(person_name)-1)
    ) AS person_name,

    CONCAT(
        SUBSTRING(date_of_birth,1,4),
        '-XX-XX'
    ) AS date_of_birth

FROM db_healthcare_kl.omop.person_data;

In [0]:
%sql

SELECT *
FROM db_healthcare_kl.omop.person_data_masked
LIMIT 10;

In [0]:
%sql

CREATE OR REPLACE VIEW db_healthcare_kl.gold.readmission_predictions_masked AS

SELECT
    SHA2(person_id, 256) AS person_id,

    total_visits,
    max_duration_days,
    avg_duration_days,
    age,
    gender_concept_id,
    diagnosis_count,
    medication_count,
    previous_admissions,
    visit_type,

    CONCAT(SUBSTRING(primary_diagnosis,1,3),'***') AS primary_diagnosis,

    Cohort_name,

    SHA2(visit_occurrence_id,256) AS visit_occurrence_id,

    visit_end_date,
    next_visit_start_date,

    readmission_30day_flag,

    processed_timestamp,

    prediction

FROM db_healthcare_kl.gold.readmission_predictions;

In [0]:
%sql

SELECT *
FROM db_healthcare_kl.gold.readmission_predictions_masked
LIMIT 10;